In [2]:
import pandas as pd
import numpy as np
import glob
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error

# Reaggregating the data
labels = pd.read_parquet('../fastf1_data/labeled/labels_combined.parquet')
segmented_files = glob.glob('../fastf1_data/processed/corners_*.parquet')
segmented = pd.concat([pd.read_parquet(f) for f in segmented_files], ignore_index=True)

target_cols = ['aggression_score', 'line_shape_score', 'oversteer_preference_score']
SAFE_COLUMNS = ['Throttle', 'Brake', 'Speed', 'nGear', 'RPM']  

# Find the length to truncate corners to
seq_lengths_used = []
for (year, race, session_type, driver, lap_num, corner_num), group in segmented.groupby(
    ['year', 'race', 'session_type', 'driver', 'lap_number', 'corner_number']
):
    match = labels[
        (labels.year == year) & (labels.race == race) & (labels.session_type == session_type) &
        (labels.driver == driver) & (labels.lap_number == lap_num) & (labels.corner_number == corner_num)
    ]
    if match.empty:
        continue
    seq_lengths_used.append(len(group))

import pandas as pd
print(pd.Series(seq_lengths_used).describe())

lengths = segmented.groupby(['year', 'race', 'session_type', 'driver', 'lap_number', 'corner_number']).size()
print(lengths.describe())
print(lengths.sort_values(ascending=False).head(10))

lengths_sorted = lengths.sort_values(ascending=False)
print(lengths_sorted.head(10))

MAX_SEQ_LEN = 100   # Covers most corners (75th percentile = 85); longer sequences are
                    # truncated to their first MAX_SEQ_LEN samples. A small number of
                    # corners (especially corner 1, sometimes 18) have anomalously long
                    # sequences — likely a slicing bug in corner_slicer.py (see notes),
                    # not real corner telemetry. Truncating avoids letting them dominate
                    # padding/memory without excluding them.

sequences, targets, groups = [], [], []

for (year, race, session_type, driver, lap_num, corner_num), group in segmented.groupby(
    ['year', 'race', 'session_type', 'driver', 'lap_number', 'corner_number']
):
    match = labels[
        (labels.year == year) &
        (labels.race == race) &
        (labels.session_type == session_type) &
        (labels.driver == driver) &
        (labels.lap_number == lap_num) &
        (labels.corner_number == corner_num)
    ]
    if match.empty:
        continue  # Corner instance didn't survive the out-lap filter in labeling

    seq = group.sort_values('Distance')[SAFE_COLUMNS].to_numpy(dtype=np.float64)

    if len(seq) >= MAX_SEQ_LEN:
        seq = seq[:MAX_SEQ_LEN]
    else:
        pad = np.zeros((MAX_SEQ_LEN - len(seq), len(SAFE_COLUMNS)))
        seq = np.vstack([seq, pad])

    sequences.append(seq)
    targets.append(match[target_cols].values[0])
    groups.append(driver)

X = np.array(sequences)   # shape: (n_corners, MAX_SEQ_LEN, n_features)
y = np.array(targets)     # shape: (n_corners, 3)
groups = np.array(groups)

print(f"X shape: {X.shape}, y shape: {y.shape}")


count    42573.000000
mean        20.902332
std          6.412362
min          5.000000
25%         15.000000
50%         20.000000
75%         25.000000
max         51.000000
dtype: float64
count    60542.000000
mean        22.821496
std         30.449462
min          5.000000
25%         16.000000
50%         21.000000
75%         26.000000
max       2131.000000
dtype: float64
year  race         session_type  driver  lap_number  corner_number
2024  Silverstone  Q             4       10.0        1                2131
                                 81      10.0        1                2043
                                 4       18.0        1                1817
                                 2       6.0         18               1732
                                 55      19.0        1                1696
                                 81      19.0        1                1666
                                 23      5.0         18               1663
                          

In [3]:
import fastf1

session = fastf1.get_session(2024, 'Italy', 'Q')  # match one of your actual ingested races
session.load(telemetry=True, laps=True)

circuit_info = session.get_circuit_info()
print(circuit_info.corners[['Number', 'Letter', 'Distance']].sort_values('Number'))

raw = pd.read_csv('../fastf1_data/raw/2024_Monza_Q.csv')  # match your actual filename
lap = raw[(raw.driver == 81) & (raw.lap_number == 10)].sort_values('Distance')

print(lap['Distance'].describe())
print(lap['Distance'].is_monotonic_increasing)

req         WARNING 	DEFAULT CACHE ENABLED! (144.82 MB) C:\Users\baohu\AppData\Local\Temp\fastf1


KeyboardInterrupt: 

In [4]:
from sklearn.model_selection import GroupShuffleSplit

# Split by DRIVER (not randomly) — prevents the model from
# seeing the same driver's data in both train and validation
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, val_idx = next(splitter.split(X, groups=groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

# Scale features (fit on train only)
# X_train shape is (n_corners, seq_len, n_features) — flatten across corners+time
# to compute per-channel mean/std, then reshape back
n_train, seq_len, n_feat = X_train.shape
flat_train = X_train.reshape(-1, n_feat)

mean = flat_train.mean(axis=0)
std = flat_train.std(axis=0)
std[std == 0] = 1e-8  # avoid divide-by-zero on any constant channel

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std

Train: (33592, 100, 5), Val: (8981, 100, 5)


In [5]:
class CornerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(CornerDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(CornerDataset(X_val, y_val), batch_size=32)


class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden_size=32, n_outputs=3):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_outputs)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)   # h_n: final hidden state after the full sequence
        return self.fc(h_n[-1])


model = LSTMRegressor(n_features=len(SAFE_COLUMNS))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

n_epochs = 50

for epoch in range(n_epochs):
    model.train()
    total_train_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * xb.size(0)
    avg_train_loss = total_train_loss / len(train_loader.dataset)

    # quick validation pass each epoch, no gradient tracking needed
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            preds = model(xb)
            loss = criterion(preds, yb)
            total_val_loss += loss.item() * xb.size(0)
    avg_val_loss = total_val_loss / len(val_loader.dataset)
    print(f"Epoch {epoch+1}/{n_epochs} — train MSE: {avg_train_loss:.4f}, val MSE: {avg_val_loss:.4f}")
    
model.eval()
with torch.no_grad():
    val_preds = model(torch.tensor(X_val, dtype=torch.float32)).numpy()

mae = mean_absolute_error(y_val, val_preds)
print(f"LSTM MAE: {mae:.4f}")
for i, col in enumerate(target_cols):
    col_mae = mean_absolute_error(y_val[:, i], val_preds[:, i])
    print(f"{col} MAE: {col_mae:.4f}")

torch.save(model.state_dict(), '../models/lstm_model.pt')

Epoch 1/50 — train MSE: 0.0761, val MSE: 0.0715
Epoch 2/50 — train MSE: 0.0701, val MSE: 0.0708
Epoch 3/50 — train MSE: 0.0701, val MSE: 0.0709
Epoch 4/50 — train MSE: 0.0700, val MSE: 0.0706
Epoch 5/50 — train MSE: 0.0700, val MSE: 0.0706
Epoch 6/50 — train MSE: 0.0699, val MSE: 0.0707
Epoch 7/50 — train MSE: 0.0700, val MSE: 0.0706
Epoch 8/50 — train MSE: 0.0699, val MSE: 0.0709
Epoch 9/50 — train MSE: 0.0699, val MSE: 0.0706
Epoch 10/50 — train MSE: 0.0699, val MSE: 0.0708
Epoch 11/50 — train MSE: 0.0699, val MSE: 0.0706
Epoch 12/50 — train MSE: 0.0699, val MSE: 0.0709


KeyboardInterrupt: 